# Explore a city in the Street View store

Pick a city that already exists under `data/<slug>/` and inspect what its
`street_data.h5` contains: summary statistics, coverage maps, image dates, and a
**spot-check** that renders a point's four images next to links that open the
*real* Google Street View at the same coordinates and heading — so you can
confirm the store holds what it claims to.

Read-only: this notebook never downloads, modifies, or rebuilds anything.

**How to use:** run cell 1 to list available cities, set `CITY` in cell 3 (or
leave it `None` to take the first one), then run top to bottom. Re-run the
spot-check cell repeatedly to see different random points.

In [ ]:
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML

import paths as paths_mod
from paths import CityPaths, slugify, DATA_DIR


def available_cities(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    """Scan data/<slug>/ for built stores and report a one-line summary each."""
    rows = []
    for d in sorted(Path(data_dir).glob("*")):
        h5 = d / "street_data.h5"
        if not h5.exists():
            continue
        try:
            with h5py.File(h5, "r") as f:
                present = f["images_present"][:]
                rows.append({
                    "slug": d.name,
                    "city": f.attrs.get("city", d.name),
                    "rows": int(f["point_id"].shape[0]),
                    "with_image": int(present.any(axis=1).sum()),
                    "total_images": int(present.sum()),
                    "image_size": f.attrs.get("image_size", "?"),
                    "h5_GB": round(h5.stat().st_size / 1e9, 2),
                    "built_at": f.attrs.get("built_at", "?"),
                })
        except Exception as ex:
            rows.append({"slug": d.name, "city": f"(unreadable: {ex})"})
    if not rows:
        print(f"No built stores found under {data_dir} (looked for */street_data.h5).")
    return pd.DataFrame(rows)


cities = available_cities()
cities

## Choose a city

Set `CITY` to a city name (e.g. `"Manchester, UK"`) or a slug (e.g.
`"manchester_uk"`). Leave it `None` to take the first store listed above.

In [ ]:
#CITY = "manchester_uk"   # city name or slug; None = first available
CITY = "bradford_uk"   # city name or slug; None = first available

if CITY is None:
    if cities.empty:
        raise RuntimeError("No stores to explore.")
    slug = cities.iloc[0]["slug"]
else:
    # Accept either an existing slug/folder or a free-form city name.
    slug = CITY if (DATA_DIR / CITY).exists() else slugify(CITY)

paths = CityPaths(slug=slug, root=DATA_DIR / slug)
H5 = paths.h5
if not H5.exists():
    raise FileNotFoundError(f"No store at {H5}. Available: {list(cities['slug'])}")
print(f"Exploring slug={slug!r}\n  H5: {H5}")

## Load the store

Everything except the raw JPEG bytes is pulled into a tidy per-point DataFrame.
Older imported stores may not have every extra dataset (e.g. `pano_lat`), so
each is loaded defensively.

In [ ]:
def _decode(col):
    return [x.decode("ascii", "ignore").rstrip("\x00") for x in col]

with h5py.File(H5, "r") as f:
    attrs = dict(f.attrs)
    present = f["images_present"][:]
    n_rows, n_h = present.shape
    headings = [int(h) for h in np.asarray(attrs.get("headings", list(range(n_h))))]

    df = pd.DataFrame({
        "point_id": f["point_id"][:],
        "lat": f["latitude"][:],
        "lon": f["longitude"][:],
        "n_images": present.sum(axis=1).astype(int),
        "date": _decode(f["date"][:]),
    })
    for name, dataset in [("status", "status"), ("round_id", "round_id"),
                          ("pano_lat", "pano_lat"), ("pano_lon", "pano_lon")]:
        if dataset in f:
            vals = f[dataset][:]
            df[name] = _decode(vals) if vals.dtype.kind == "S" else vals

print(f"{n_rows:,} points | {n_h} headings {headings} | {int(present.sum()):,} images")
print(f"City attr: {attrs.get('city', '?')!r}  image_size: {attrs.get('image_size', '?')}")
df.head()

## Summary statistics

In [ ]:
n_any = int((df.n_images > 0).sum())
print(f"Points with >=1 image : {n_any:,} / {n_rows:,} ({100*n_any/n_rows:.1f}%)")
print(f"Points with all {n_h}    : {int((df.n_images == n_h).sum()):,}")
print(f"Points with 0 images  : {int((df.n_images == 0).sum()):,}")
print()
print("Images per point:")
print(df.n_images.value_counts().sort_index().to_string())

# Per-heading presence — which compass directions are missing most.
print("\nPresence per heading:")
for j, h in enumerate(headings):
    c = int(present[:, j].sum())
    print(f"  {h:>3}deg : {c:,} ({100*c/n_rows:.1f}%)")

if "status" in df:
    print("\nPano status breakdown:")
    print(df.status.value_counts().to_string())
if "round_id" in df:
    print("\nPoints per sampling round:")
    print(df.round_id.value_counts().sort_index().to_string())

In [ ]:
# Sampling-round manifest, if present (records spacing + when each round ran).
if paths.samples_manifest.exists():
    display(pd.read_parquet(paths.samples_manifest))
else:
    print("No samples_manifest.parquet for this city.")

In [ ]:
# On-disk JPEG cache vs what the H5 says is present (should agree).
if paths.images_dir.exists():
    jpgs = list(paths.images_dir.glob("*.jpg"))
    size_gb = sum(p.stat().st_size for p in jpgs) / 1e9
    print(f"JPEGs on disk      : {len(jpgs):,} files, {size_gb:.2f} GB")
    print(f"images_present (H5): {int(present.sum()):,}")
    if len(jpgs) != int(present.sum()):
        print("  note: counts differ — disk cache and H5 are out of sync.")
else:
    print("No street_images/ directory (images live only inside the H5).")

## Maps

Point locations on a basemap. The first map colours by image coverage; the
others by image date and (if present) sampling round. Needs internet for the
basemap tiles — if that fails the points still plot on a blank background.

In [ ]:
import contextily as cx
import geopandas as gpd

gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326
).to_crs(3857)

# Year extracted from the date string (YYYY-MM or YYYY-MM-DD) for colouring.
gdf["year"] = df.date.str.slice(0, 4).replace("", "unknown")


def plot_points(column, title, cmap="viridis", categorical=False, markersize=4):
    fig, ax = plt.subplots(figsize=(11, 11))
    gdf.plot(ax=ax, column=column, cmap=cmap, markersize=markersize,
             legend=True, categorical=categorical,
             legend_kwds=({"loc": "upper right", "markerscale": 2,
                           "fontsize": 8} if categorical else {"shrink": 0.5}))
    if paths.boundary_geojson.exists():
        gpd.read_file(paths.boundary_geojson).to_crs(3857).boundary.plot(
            ax=ax, edgecolor="red", linewidth=1)
    try:
        cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
    except Exception as ex:
        print("basemap unavailable, plotting without tiles:", ex)
    ax.set_axis_off()
    ax.set_title(f"{slug} — {title}")
    plt.tight_layout()
    plt.show()


plot_points("n_images", f"images per point (0–{n_h})", cmap="viridis")

In [ ]:
# Colour by capture year — reveals patches of older/newer Street View imagery.
plot_points("year", "Street View capture year", cmap="tab10", categorical=True)

In [ ]:
# Colour by sampling round (if the city was sampled in more than one pass).
if "round_id" in gdf and gdf.round_id.nunique() > 1:
    plot_points("round_id", "sampling round", cmap="Set1", categorical=True)
else:
    print("Single sampling round — nothing to compare.")

## Image dates

In [ ]:
# Distribution of capture dates (year-month). Blank = no date recorded.
ym = df.date.str.slice(0, 7)
counts = ym[ym != ""].value_counts().sort_index()
if len(counts):
    ax = counts.plot(kind="bar", figsize=(14, 4), color="steelblue")
    ax.set_title(f"{slug} — Street View imagery by year-month")
    ax.set_xlabel("")
    ax.set_ylabel("points")
    n_blank = int((ym == "").sum())
    if n_blank:
        print(f"{n_blank:,} points have no recorded date.")
    plt.tight_layout()
    plt.show()
else:
    print("No dates recorded for this city.")

## Spot check against real Street View

Renders one point's four headings, then prints links that open Google Street
View at the **same coordinates and heading**. Compare the rendered image to what
Google shows — they should match (allowing for Google having since refreshed the
panorama).

Set `POINT_ID` to inspect a specific point; leave it `None` for a random point
that has all four images. **Re-run this cell** to draw a fresh random point.

In [ ]:
POINT_ID = None  # int to pin a point; None = random fully-imaged point

with h5py.File(H5, "r") as f:
    pids = f["point_id"][:]
    if POINT_ID is None:
        cand = np.where(present.all(axis=1))[0]
        if len(cand) == 0:
            cand = np.where(present.any(axis=1))[0]
        row = int(np.random.choice(cand))
    else:
        hits = np.where(pids == POINT_ID)[0]
        if len(hits) == 0:
            raise ValueError(f"point_id {POINT_ID} not in this store")
        row = int(hits[0])

    pid = int(pids[row])
    plat, plon = float(f["latitude"][row]), float(f["longitude"][row])
    d = f["date"][row].decode("ascii", "ignore").rstrip("\x00")
    pano = (f["pano_id"][row].decode("utf-8", "ignore").rstrip("\x00")
            if "pano_id" in f else "")
    imgs = [Image.open(io.BytesIO(f["images_jpeg"][row, j].tobytes())).convert("RGB")
            if present[row, j] else None for j in range(n_h)]

fig, axes = plt.subplots(1, n_h, figsize=(4 * n_h, 4))
for ax, h, im in zip(np.atleast_1d(axes), headings, imgs):
    if im is None:
        ax.set_title(f"{h}° (missing)")
    else:
        ax.imshow(im)
        ax.set_title(f"{h}°")
    ax.axis("off")
plt.suptitle(f"point_id={pid}   ({plat:.6f}, {plon:.6f})   date={d or '?'}")
plt.tight_layout()
plt.show()

links = "".join(
    f'<li>heading {h}°: '
    f'<a href="https://www.google.com/maps/@?api=1&map_action=pano'
    f'&viewpoint={plat},{plon}&heading={h}" target="_blank">open in Street View</a></li>'
    for h in headings
)
display(HTML(
    f"<b>Compare against the real Street View</b> — point {pid} "
    f"({plat:.6f}, {plon:.6f}){', pano ' + pano if pano else ''}"
    f"<ul>{links}</ul>"
    f'<a href="https://www.google.com/maps/search/?api=1&query={plat},{plon}" '
    f'target="_blank">Open this location in Google Maps</a>'
))

## Quick scan — many points at once

A grid of the heading-0 image for a sample of random points, each with its
coordinates and a Street View link. Good for eyeballing a batch quickly.

In [ ]:
N_SCAN = 12  # how many random points to show

cand = np.where(present[:, 0])[0]  # rows that have a heading-0 image
sample = np.random.choice(cand, size=min(N_SCAN, len(cand)), replace=False)
sample = sorted(int(r) for r in sample)

ncols = 4
nrows = int(np.ceil(len(sample) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = np.atleast_1d(axes).ravel()

link_rows = []
with h5py.File(H5, "r") as f:
    for ax, row in zip(axes, sample):
        pid = int(f["point_id"][row])
        plat, plon = float(f["latitude"][row]), float(f["longitude"][row])
        img = Image.open(io.BytesIO(f["images_jpeg"][row, 0].tobytes())).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"pid {pid}\n{plat:.5f}, {plon:.5f}", fontsize=9)
        ax.axis("off")
        link_rows.append(
            f'<li>pid {pid} ({plat:.6f}, {plon:.6f}): '
            f'<a href="https://www.google.com/maps/@?api=1&map_action=pano'
            f'&viewpoint={plat},{plon}&heading={headings[0]}" target="_blank">Street View</a></li>'
        )
for ax in axes[len(sample):]:
    ax.axis("off")
plt.suptitle(f"{slug} — heading {headings[0]}° for {len(sample)} random points")
plt.tight_layout()
plt.show()
display(HTML("<ul>" + "".join(link_rows) + "</ul>"))